In [8]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv('../data/df_with_2regimes.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3518 entries, 0 to 3517
Data columns (total 37 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   date                                        3518 non-null   object 
 1   log_return                                  3518 non-null   float64
 2   realized_variance                           3518 non-null   float64
 3   realized_volatility                         3518 non-null   float64
 4   gtrend_pct_change                           3518 non-null   float64
 5   blockchain_diff_log_n_transactions          3518 non-null   float64
 6   blockchain_diff_log_transaction_fee_usd     3518 non-null   float64
 7   blockchain_diff_log_n_unique_addresses      3518 non-null   float64
 8   blockchain_diff_log_transaction_volume_usd  3518 non-null   float64
 9   log_volume                                  3518 non-null   float64
 10  target      

In [10]:
# lag 1,2 andd 3 of realized volatility
df['rv_lag1'] = df['realized_volatility'].shift(1)
df['rv_lag2'] = df['realized_volatility'].shift(2)
df['rv_lag3'] = df['realized_volatility'].shift(3)

# rolling mean of realized volatility over the past 30 days
df['rv_rolling_mean_30'] = df['realized_volatility'].rolling(window=30).mean() 

df.head(30)

,date,log_return,realized_variance,realized_volatility,gtrend_pct_change,blockchain_diff_log_n_transactions,blockchain_diff_log_transaction_fee_usd,blockchain_diff_log_n_unique_addresses,blockchain_diff_log_transaction_volume_usd,log_volume,...,blockchain_log_mempool_count_chg,p_sigma,p_r,p_d,regime,regime_id,rv_lag1,rv_lag2,rv_lag3,rv_rolling_mean_30
0,2016-06-15,0.019990,0.000528,0.022974,-0.138889,0.048912,0.154753,0.114522,-0.141196,8.976393,...,-0.003401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-06-16,0.096743,0.001641,0.040503,0.225806,0.032894,0.084044,0.034834,0.704898,9.729602,...,-0.060516,NaN,NaN,NaN,NaN,NaN,0.022974,NaN,NaN,NaN
2,2016-06-17,-0.022997,0.003730,0.061075,0.039474,-0.054144,-0.035212,-0.034919,-0.362618,9.547441,...,-0.003399,NaN,NaN,NaN,NaN,NaN,0.040503,0.022974,NaN,NaN
3,2016-06-18,0.008907,0.001710,0.041354,-0.177215,-0.075424,-0.124417,-0.069753,-0.418154,9.169094,...,-0.033404,NaN,NaN,NaN,NaN,NaN,0.061075,0.040503,0.022974,NaN
4,2016-06-19,0.012790,0.000586,0.024214,-0.076923,-0.075072,-0.097766,-0.024354,-0.288327,8.602203,...,-0.132230,NaN,NaN,NaN,NaN,NaN,0.041354,0.061075,0.040503,NaN
5,2016-06-20,-0.089251,0.002527,0.050274,0.116667,0.132932,0.218720,0.113885,0.428680,9.194033,...,0.101567,NaN,NaN,NaN,NaN,NaN,0.024214,0.041354,0.061075,NaN
6,2016-06-21,-0.051749,0.010900,0.104401,0.119403,-0.004704,-0.070244,0.001482,0.406474,10.170138,...,-0.037078,NaN,NaN,NaN,NaN,NaN,0.050274,0.024214,0.041354,NaN
7,2016-06-22,-0.099377,0.005286,0.072708,-0.186667,-0.051780,-0.070697,-0.122028,-0.102793,9.654605,...,0.073030,NaN,NaN,NaN,NaN,NaN,0.104401,0.050274,0.024214,NaN
8,2016-06-23,0.043063,0.013669,0.116915,0.114754,0.023732,-0.157445,0.068596,-0.121465,9.926960,...,-0.022153,NaN,NaN,NaN,NaN,NaN,0.072708,0.104401,0.050274,NaN
9,2016-06-24,0.059784,0.010043,0.100214,0.470588,-0.015347,0.093934,-0.010400,-0.264467,9.856388,...,0.044270,NaN,NaN,NaN,NaN,NaN,0.116915,0.072708,0.104401,NaN


In [11]:
# drop rows with NaN values (due to lag and rolling mean)
df = df.dropna().reset_index(drop=True)

print(df.isna().sum())

date                                          0
log_return                                    0
realized_variance                             0
realized_volatility                           0
gtrend_pct_change                             0
blockchain_diff_log_n_transactions            0
blockchain_diff_log_transaction_fee_usd       0
blockchain_diff_log_n_unique_addresses        0
blockchain_diff_log_transaction_volume_usd    0
log_volume                                    0
target                                        0
gold_close_ret                                0
silver_close_ret                              0
brent_close_ret                               0
dji_close_ret                                 0
spx_close_ret                                 0
rut_close_ret                                 0
nasdaq_close_ret                              0
usdcny_close_ret                              0
usdeur_close_ret                              0
gold_volume_chg                         

In [12]:
x_features = ["gtrend_pct_change",
    "blockchain_diff_log_n_transactions",
    "blockchain_diff_log_transaction_fee_usd",
    "blockchain_diff_log_n_unique_addresses",
    "blockchain_diff_log_transaction_volume_usd",
    "log_volume",
    "gold_close_ret",
    "silver_close_ret",
    "brent_close_ret",
    "dji_close_ret",
    "spx_close_ret",
    "rut_close_ret",
    "nasdaq_close_ret",
    "usdcny_close_ret",
    "usdeur_close_ret",
    "gold_volume_chg",
    "silver_volume_chg",
    "brent_volume_chg",
    "dji_volume_chg",
    "spx_volume_chg",
    "rut_volume_chg",
    "nasdaq_volume_chg",
    "vix_close_chg",
    "hash-rate_chg",
    "difficulty_chg",
    "median-confirmation-time_chg",
    "blockchain_log_mempool_count_chg", 
    "rv_lag1",
    "rv_lag2",
    "rv_lag3",
    "rv_rolling_mean_30"
    ]

print(len(x_features))

31


In [13]:
train_size = int(len(df) * 0.7)
eval_size = int(len(df) * 0.15)
train_df = df.iloc[:train_size]

eval_df = df.iloc[train_size:train_size + eval_size]
test_df = df.iloc[train_size + eval_size:]
print(f"Train set: {len(train_df)} rows")
print(f"Evaluation set: {len(eval_df)} rows")
print(f"Test set: {len(test_df)} rows")


X_train_lstm = train_df[x_features]
X_eval_lstm  = eval_df[x_features]
X_test_lstm  = test_df[x_features]

y_train_regime = train_df["regime_id"].shift(-1)  # shift by 1 to predict next day's regime
y_eval_regime  = eval_df["regime_id"]
y_test_regime  = test_df["regime_id"]

Train set: 2442 rows
Evaluation set: 523 rows
Test set: 524 rows


In [14]:
def compute_weights(regimes, lam=0.5, eps=1e-12):
    regimes = np.asarray(regimes).astype(int)
    T = len(regimes)

    switch_flag = np.zeros(T, dtype=int)
    switch_flag[1:] = (regimes[1:] != regimes[:-1]).astype(int)

    unique_regimes = np.unique(regimes)

    class_count = {g: np.sum(regimes == g) for g in unique_regimes}
    switch_count = {
        g: np.sum((regimes == g) & (switch_flag == 1))
        for g in unique_regimes
    }
    total_switches = np.sum(switch_flag)

    weights = np.zeros(T, dtype=float)

    for t in range(T):
        g = regimes[t]

        imbalance_term = T / (class_count[g] + eps)

        if total_switches > 0 and switch_count[g] > 0:
            switch_term = total_switches / (switch_count[g] + eps)
        else:
            switch_term = 0.0 

        weights[t] = (1 - lam) * imbalance_term + lam * switch_term 

    # normalize weights for stability
    weights = weights / np.mean(weights)

    return weights, switch_flag

In [15]:
# use this for evaluation and testing
def create_rolling_scaled_sequences_range(
    df, feature_cols, target_col, lookback, scale_window, start_idx, end_idx, date_col=None
):
    X, y, dates = [], [], []

    first_valid_idx = max(start_idx, lookback, scale_window)

    for t in range(first_valid_idx, end_idx - 1):

        hist_window = df[feature_cols].iloc[t - scale_window:t]
        seq_window = df[feature_cols].iloc[t - lookback:t]

        target = df[target_col].iloc[t + 1]

        if hist_window.isnull().any().any():
            continue
        if seq_window.isnull().any().any():
            continue
        if pd.isna(target):
            continue

        scaler = StandardScaler()
        scaler.fit(hist_window)

        seq_scaled = scaler.transform(seq_window)

        X.append(seq_scaled)
        y.append(target)

        if date_col is not None:
            dates.append(df[date_col].iloc[t + 1])
        else:
            dates.append(t + 1)

    return np.array(X), np.array(y), dates

In [16]:
# use this for training
def create_rolling_scaled_sequences_range_weights(
    df, feature_cols, target_col, lookback, scale_window,
    start_idx, end_idx, weights_full, date_col=None
):
    X, y, dates, target_idx, w = [], [], [], [], []

    first_valid_idx = max(start_idx, lookback, scale_window)

    for t in range(first_valid_idx, end_idx - 1):
        hist_window = df[feature_cols].iloc[t - scale_window:t]
        seq_window = df[feature_cols].iloc[t - lookback:t]
        target = df[target_col].iloc[t + 1]

        # weight aligned with target at t+1
        weight = weights_full[t + 1]

        if hist_window.isnull().any().any():
            continue
        if seq_window.isnull().any().any():
            continue
        if pd.isna(target):
            continue
        if pd.isna(weight):
            continue

        scaler = StandardScaler()
        scaler.fit(hist_window)
        seq_scaled = scaler.transform(seq_window)

        X.append(seq_scaled)
        y.append(target)
        w.append(weight)
        target_idx.append(t + 1)

        if date_col is not None:
            dates.append(df[date_col].iloc[t + 1])
        else:
            dates.append(t + 1)

    return np.array(X), np.array(y), dates, np.array(w), np.array(target_idx)

def scale_rolling(df_hist, feature_cols, lookback, scale_window=730):

    scaler_hist = df_hist[feature_cols].iloc[-min(scale_window, len(df_hist)):].copy()

    scaler = StandardScaler()
    scaler.fit(scaler_hist)

    seq_raw = df_hist[feature_cols].iloc[-lookback:].copy()

    seq_scaled = scaler.transform(seq_raw)

    return seq_scaled, scaler

def build_lstm_model(lookback, n_features, n_regimes=2, units=64, dropout=0.2, optimizer="adam"):
    model = Sequential([
        LSTM(units, input_shape=(lookback, n_features)),
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(n_regimes, activation="softmax")
    ])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [17]:
def rolling_regime_forecast(
    df_full,
    lstm_model,
    feature_cols,
    lookback,
    start_idx,
    end_idx,
    scale_window=730,
    date_col="date",
    target_col="regime_id"
):

    results = []

    for t in range(start_idx - 1, end_idx - 1):

        df_hist = df_full.iloc[:t+1].copy()

        if len(df_hist) < lookback:
            continue

        try:
            seq_scaled, _ = scale_rolling(
                df_hist,
                feature_cols=feature_cols,
                lookback=lookback,
                scale_window=scale_window
            )
        except Exception:
            continue

        X_input = np.expand_dims(seq_scaled, axis=0)

        pred_probs = lstm_model.predict(X_input, verbose=0)
        pred_regime = int(np.argmax(pred_probs))

        next_row = df_full.iloc[t+1]

        results.append({
            "date": next_row[date_col],
            "pred_regime": pred_regime,
            "pred_prob_0": pred_probs[0,0],
            "pred_prob_1": pred_probs[0,1]
        })

    return pd.DataFrame(results)

In [18]:
min_garch_obs = 200

def fit_garch_safe(returns, p=1, o=1, q=1, dist="normal", vol_model="GARCH"):
    returns = returns.dropna().astype(float)

    if len(returns) < min_garch_obs:
        return None

    try:
        vol_model = vol_model.upper()

        kwargs = dict(
            y=returns * 100.0,
            mean="constant",
            vol=vol_model,
            p=p,
            q=q,
            dist=dist
        )

        if vol_model == "EGARCH":
            kwargs["o"] = o

        am = arch_model(**kwargs)
        res = am.fit(disp="off")
        return res

    except Exception:
        return None


def one_step_var_forecast(fitted_vol_model):
    if fitted_vol_model is None:
        return np.nan

    try:
        fcast = fitted_vol_model.forecast(horizon=1, reindex=False)
        var_100 = fcast.variance.values[-1, 0]
        return var_100 / (100.0 ** 2)
    except Exception:
        return np.nan

In [19]:
def rolling_lstm_garch_soft_eval(
    df_full,
    regime_forecast_df,
    start_idx,
    end_idx,
    return_col="log_return",
    realized_var_col="realized_variance",
    regime_col="regime_id",
    date_col="date",
    garch_p=1,
    garch_o=1,
    garch_q=1,
    vol_model="GARCH",
    garch_dist="normal",
    rolling_window=730
):
    results = []

    regime_map = regime_forecast_df.set_index(date_col)

    for t in range(start_idx - 1, end_idx - 1):
        next_row = df_full.iloc[t + 1]
        next_date = next_row[date_col]

        if next_date not in regime_map.index:
            continue

        regime_pred_row = regime_map.loc[next_date]

        
        if isinstance(regime_pred_row, pd.DataFrame):
            regime_pred_row = regime_pred_row.iloc[0]

        window_df = df_full.iloc[max(0, t - rolling_window + 1): t + 1].copy()

        low_returns = window_df.loc[window_df[regime_col] == 0, return_col].dropna()
        high_returns = window_df.loc[window_df[regime_col] == 1, return_col].dropna()

        garch_low = fit_garch_safe(
            low_returns,
            p=garch_p,
            o=garch_o,
            q=garch_q,
            vol_model=vol_model,
            dist=garch_dist
        )

        garch_high = fit_garch_safe(
            high_returns,
            p=garch_p,
            o=garch_o,
            q=garch_q,
            vol_model=vol_model,
            dist=garch_dist
        )

        var_low = one_step_var_forecast(garch_low)
        var_high = one_step_var_forecast(garch_high)

        p0 = float(regime_pred_row["pred_prob_0"])
        p1 = float(regime_pred_row["pred_prob_1"])

        if np.isfinite(var_low) and np.isfinite(var_high):
            var_soft = p0 * var_low + p1 * var_high
        else:
            var_soft = np.nan

        results.append({
            "date": next_date,
            "actual_var": next_row[realized_var_col],
            "true_regime": int(next_row[regime_col]),
            "pred_regime": int(regime_pred_row["pred_regime"]),
            "pred_prob_0": p0,
            "pred_prob_1": p1,
            "var_low": var_low,
            "var_high": var_high,
            "var_soft": var_soft
        })

    return pd.DataFrame(results)

In [20]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))


def qlike(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true > 0) & (y_pred > 0)
    if mask.sum() == 0:
        return np.nan

    yt = y_true[mask]
    yp = y_pred[mask]

    return np.mean(np.log(yp + eps) + yt / (yp + eps))


In [21]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras import backend as K
from itertools import product 
from arch import arch_model
import tensorflow as tf
import gc

In [22]:
train_start_idx = 0
train_end_idx = train_size

eval_start_idx = train_size
eval_end_idx = train_size + eval_size

test_start_idx = eval_end_idx
test_end_idx = len(df)

In [23]:
lstm_grid = {
    "lookback": [20, 30, 60],
    "units": [32, 64],
    "dropout": [0.0, 0.01, 0.1, 0.2],
    "batch_size": [32]
} 

garch_grid = {
    "model": ["EGARCH"],
    "p": [1,2],
    "q": [1,2],
    "o": [1],
    "dist": ["t"]
}

param_grid = []

for lb, units, drop, bs, vol, p, q, o, dist in product(
    lstm_grid["lookback"],
    lstm_grid["units"],
    lstm_grid["dropout"],
    lstm_grid["batch_size"],
    garch_grid["model"],
    garch_grid["p"],
    garch_grid["q"],
    garch_grid["o"],
    garch_grid["dist"]
):
    param_grid.append({
        "lookback": lb,
        "units": units,
        "dropout": drop,
        "batch_size": bs,
        "garch_model": vol,  
        "garch_p": p,
        "garch_q": q,
        "garch_o": o,
        "garch_dist": dist
    })

print(f"Total hyperparameter combinations: {len(param_grid)}")

Total hyperparameter combinations: 96


In [24]:
tuning_results = []

for i, params in enumerate(param_grid, 1):
    print(f"\n[{i}/{len(param_grid)}] Testing params: {params}")

    K.clear_session()
    tf.keras.backend.clear_session()
    gc.collect()

    lookback = params["lookback"]

    # compute weights from training set
    train_weights_full, train_switch_flags = compute_weights(
        train_df["regime_id"].values, lam=0.5
    )

    # rolling-scaled training sequences
    X_train_lstm, y_train_regime, train_dates, w_train, train_idx = (
        create_rolling_scaled_sequences_range_weights(
            df=train_df,
            feature_cols=x_features,
            target_col="regime_id",
            lookback=lookback,
            scale_window=730,
            start_idx=0,
            end_idx=len(train_df),
            weights_full=train_weights_full,
            date_col="date"
        )
    )

    w_train = w_train.astype(np.float32)

    # rolling-scaled evaluation sequences
    X_eval_lstm, y_eval_regime, eval_dates = create_rolling_scaled_sequences_range(
        df=df,
        feature_cols=x_features,
        target_col="regime_id",
        lookback=lookback,
        scale_window=730,
        start_idx=eval_start_idx,
        end_idx=eval_end_idx,
        date_col="date"
    )

    if len(X_train_lstm) == 0 or len(X_eval_lstm) == 0:
        print("Skipping: no sequences produced.")
        continue

    y_train_cat = to_categorical(y_train_regime, num_classes=2)
    y_eval_cat = to_categorical(y_eval_regime, num_classes=2)

    model = build_lstm_model(
        lookback=lookback,
        n_features=len(x_features),
        n_regimes=2,
        units=params["units"],
        dropout=params["dropout"],
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    model.fit(
        X_train_lstm,
        y_train_cat,
        sample_weight=w_train,
        validation_data=(X_eval_lstm, y_eval_cat),
        epochs=100,
        batch_size=params["batch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    # rolling regime forecast on evaluation period
    eval_regime_roll = rolling_regime_forecast(
        df_full=df,
        lstm_model=model,
        feature_cols=x_features,
        lookback=lookback,
        start_idx=eval_start_idx,
        end_idx=eval_end_idx,
        scale_window=730,
        date_col="date",
        target_col="regime_id"
    )

    if eval_regime_roll.empty:
        print("Skipping: rolling regime forecast returned empty dataframe.")
        continue

    # stage 2: regime-aware GARCH/EGARCH soft forecast
    eval_stage2 = rolling_lstm_garch_soft_eval(
        df_full=df,
        regime_forecast_df=eval_regime_roll,
        start_idx=eval_start_idx,
        end_idx=eval_end_idx,
        return_col="log_return",
        realized_var_col="realized_variance",
        regime_col="regime_id",
        date_col="date",
        garch_p=params["garch_p"],
        garch_o=params["garch_o"],
        garch_q=params["garch_q"],
        vol_model=params["garch_model"],
        garch_dist=params["garch_dist"],
        rolling_window=730
    )

    if eval_stage2.empty:
        print("Skipping: stage-2 evaluation returned empty dataframe.")
        continue

    print(eval_stage2[["actual_var", "var_low", "var_high", "var_soft"]].describe())
    print(eval_stage2.loc[~np.isfinite(eval_stage2["var_soft"])])
    print(eval_stage2.sort_values("var_soft", ascending=False).head(10))

    score_rmse = rmse(eval_stage2["actual_var"], eval_stage2["var_soft"])
    score_qlike = qlike(eval_stage2["actual_var"], eval_stage2["var_soft"])

    tuning_results.append({
        **params,
        "eval_rmse": score_rmse,
        "eval_qlike": score_qlike,
        "n_eval_forecasts": len(eval_stage2)
    })

    print(f"Eval RMSE : {score_rmse:.6f}")
    print(f"Eval QLIKE: {score_qlike:.6f}")


[1/96] Testing params: {'lookback': 20, 'units': 32, 'dropout': 0.0, 'batch_size': 32, 'garch_model': 'EGARCH', 'garch_p': 1, 'garch_q': 1, 'garch_o': 1, 'garch_dist': 't'}


2026-03-20 13:03:39.810857: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/ar

       actual_var        var_low       var_high       var_soft
count  523.000000   5.230000e+02   5.230000e+02   5.230000e+02
mean     0.000604  1.374909e+302  2.406090e+302  1.774070e+302
std      0.000843            inf            inf            inf
min      0.000009   0.000000e+00   0.000000e+00   4.958172e-21
25%      0.000170   7.779860e-05   1.013572e-03   5.701322e-04
50%      0.000381   1.899931e-04   1.286316e-03   7.617535e-04
75%      0.000728   3.080606e-04   1.736107e-03   1.929121e-03
max      0.010183  1.797693e+304  1.797693e+304  9.987325e+303
Empty DataFrame
Columns: [date, actual_var, true_regime, pred_regime, pred_prob_0, pred_prob_1, var_low, var_high, var_soft]
Index: []
           date  actual_var  true_regime  pred_regime  pred_prob_0  \
119  2023-07-19    0.000168            0            0     0.555563   
170  2023-09-08    0.000205            1            1     0.467228   
121  2023-07-21    0.000079            0            1     0.469809   
316  2024-02-01   

/var/folders/2h/gvrjdw1x6t766jgk66w62_tw0000gn/T/ipykernel_1644/851364608.py:7: RuntimeWarning: overflow encountered in square
  return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/

       actual_var        var_low       var_high       var_soft
count  523.000000   5.230000e+02   5.230000e+02   5.230000e+02
mean     0.000604  2.406090e+302  3.437272e+301  1.370683e+302
std      0.000843            inf            inf            inf
min      0.000009   0.000000e+00   0.000000e+00   5.077803e-33
25%      0.000170   7.631812e-05   9.989168e-04   5.909504e-04
50%      0.000381   1.842949e-04   1.213322e-03   7.855300e-04
75%      0.000728   3.026951e-04   1.575819e-03   1.162813e-03
max      0.010183  1.797693e+304  1.797693e+304  1.019563e+304
Empty DataFrame
Columns: [date, actual_var, true_regime, pred_regime, pred_prob_0, pred_prob_1, var_low, var_high, var_soft]
Index: []
           date  actual_var  true_regime  pred_regime  pred_prob_0  \
75   2023-06-05    0.001126            1            0     0.567151   
76   2023-06-06    0.000808            1            0     0.532610   
78   2023-06-08    0.000247            1            1     0.488492   
57   2023-05-18   

/var/folders/2h/gvrjdw1x6t766jgk66w62_tw0000gn/T/ipykernel_1644/851364608.py:7: RuntimeWarning: overflow encountered in square
  return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Fram

       actual_var        var_low       var_high       var_soft
count  523.000000   5.230000e+02   5.230000e+02   5.230000e+02
mean     0.000604  4.812180e+302  6.233896e+302  5.564897e+302
std      0.000843            inf            inf            inf
min      0.000009   0.000000e+00   0.000000e+00   4.486013e-38
25%      0.000170   1.013826e-04   9.935726e-04   5.654868e-04
50%      0.000381   2.020610e-04   1.374040e-03   9.501724e-04
75%      0.000728   4.278673e-04   7.810132e-03   5.837562e-02
max      0.010183  1.797693e+304  1.797693e+304  1.200996e+304
Empty DataFrame
Columns: [date, actual_var, true_regime, pred_regime, pred_prob_0, pred_prob_1, var_low, var_high, var_soft]
Index: []
           date  actual_var  true_regime  pred_regime  pred_prob_0  \
24   2023-04-15    0.000065            0            0     0.668076   
17   2023-04-08    0.000063            0            0     0.665142   
11   2023-04-02    0.000180            0            0     0.628887   
60   2023-05-21   

/var/folders/2h/gvrjdw1x6t766jgk66w62_tw0000gn/T/ipykernel_1644/851364608.py:7: RuntimeWarning: overflow encountered in square
  return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Fram

KeyboardInterrupt: 